In [59]:
import pandas as pd
import numpy as np

In [60]:
rating_df = pd.read_csv("ratings.csv")
rating_df.head()

,userId,movieId,rating,timestamp
0,1,16,4.000,1217897793
1,1,24,1.500,1217895807
2,1,32,4.000,1217896246
3,1,47,4.000,1217896556
4,1,50,4.000,1217896523


In [61]:
rating_df.drop(columns=['timestamp'], inplace=True)
rating_df

,userId,movieId,rating
0,1,16,4.000
1,1,24,1.500
2,1,32,4.000
3,1,47,4.000
4,1,50,4.000
...,...,...,...
105334,668,142488,4.000
105335,668,142507,3.500
105336,668,143385,4.000
105337,668,144976,2.500


In [62]:
movies_df = pd.read_csv("movies.csv.xls")
movies_df.drop(columns=['genres'], inplace=True)
movies_df

,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)
...,...,...
10324,146684,Cosmic Scrat-tastrophe (2015)
10325,146878,Le Grand Restaurant (1966)
10326,148238,A Very Murray Christmas (2015)
10327,148626,The Big Short (2015)


In [63]:
df = pd.merge(movies_df, rating_df, on = 'movieId')
df.head()

,movieId,title,userId,rating
0,1,Toy Story (1995),2,5.000
1,1,Toy Story (1995),5,4.000
2,1,Toy Story (1995),8,5.000
3,1,Toy Story (1995),11,4.000
4,1,Toy Story (1995),14,4.000


In [64]:
df['title'].isna().sum()

np.int64(0)

In [65]:
mov_rating_count = (df.
    groupby(by=['title'])['rating'].
    count().
    reset_index().
    rename(columns={'rating' : 'totalRatingCount'})
    [["title",'totalRatingCount']]
                   )
mov_rating_count

,title,totalRatingCount
0,'71 (2014),1
1,'Hellboy': The Seeds of Creation (2004),1
2,'Round Midnight (1986),1
3,'Til There Was You (1997),3
4,"'burbs, The (1989)",20
...,...,...
10318,loudQUIETloud: A Film About the Pixies (2006),1
10319,xXx (2002),24
10320,xXx: State of the Union (2005),7
10321,¡Three Amigos! (1986),40


In [66]:
rating_with_totalRatingCount = df.merge(
    mov_rating_count,
    on='title',
    how='left'
)

rating_with_totalRatingCount.head()

,movieId,title,userId,rating,totalRatingCount
0,1,Toy Story (1995),2,5.000,232
1,1,Toy Story (1995),5,4.000,232
2,1,Toy Story (1995),8,5.000,232
3,1,Toy Story (1995),11,4.000,232
4,1,Toy Story (1995),14,4.000,232


In [67]:
rating_with_totalRatingCount = rating_with_totalRatingCount[
    ['userId', 'movieId', 'rating', 'title', 'totalRatingCount']
]

rating_with_totalRatingCount.head()

,userId,movieId,rating,title,totalRatingCount
0,2,1,5.000,Toy Story (1995),232
1,5,1,4.000,Toy Story (1995),232
2,8,1,5.000,Toy Story (1995),232
3,11,1,4.000,Toy Story (1995),232
4,14,1,4.000,Toy Story (1995),232


In [68]:
pd.set_option("display.float_format", lambda x: '%.3f' % x)
print(mov_rating_count['totalRatingCount'].describe())

count   10323.000
mean       10.204
std        22.835
min         1.000
25%         1.000
50%         3.000
75%         8.000
max       325.000
Name: totalRatingCount, dtype: float64


In [69]:
popularity_threshold = 50
rating_popular_movie = rating_with_totalRatingCount.query('totalRatingCount >= @popularity_threshold')
rating_popular_movie.head()

,userId,movieId,rating,title,totalRatingCount
0,2,1,5.000,Toy Story (1995),232
1,5,1,4.000,Toy Story (1995),232
2,8,1,5.000,Toy Story (1995),232
3,11,1,4.000,Toy Story (1995),232
4,14,1,4.000,Toy Story (1995),232


In [70]:
movie_matrix=rating_popular_movie.pivot_table(index="title", columns = "userId", values='rating').fillna(0)
movie_matrix.head()

userId,1,2,3,4,5,6,7,8,9,10,...,659,660,661,662,663,664,665,666,667,668
title,,,,,,,,,,,,,,,,,,,,,
10 Things I Hate About You (1999),0.000,0.000,0.000,0.000,0.000,5.000,0.000,0.000,0.000,5.000,...,2.500,0.000,0.000,0.000,0.000,0.000,0.000,4.000,0.000,2.000
12 Angry Men (1957),0.000,0.000,0.000,5.000,0.000,0.000,0.000,0.000,0.000,0.000,...,4.500,0.000,0.000,0.000,0.000,0.000,0.000,5.000,0.000,4.500
2001: A Space Odyssey (1968),0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,5.000,0.000,0.000,0.000,0.000,0.000,3.000
28 Days Later (2002),0.000,0.000,0.000,0.000,0.000,0.000,2.500,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.500
300 (2007),0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,3.500,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.500


In [71]:
from scipy.sparse import csr_matrix
movie_features_matrix = csr_matrix(movie_matrix.values)
from sklearn.neighbors import NearestNeighbors

model_knn = NearestNeighbors(metric= 'cosine', algorithm = 'brute')
model_knn.fit(movie_features_matrix)


,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [83]:
query_index = np.random.choice(movie_matrix.shape[0])
print(query_index)
distances, indices = model_knn.kneighbors(movie_matrix.iloc[query_index,:].values.reshape(1, -1), n_neighbors = 6)

2


In [84]:
for i in range(0, len(distances.flatten())):
    if i == 0:
        print('Recommendations for {0}:\n'.format(
            movie_matrix.index[query_index]
        ))
    else:
        print('{0}: {1}, with distance of {2}:'.format(
            i,
            movie_matrix.index[indices.flatten()[i]],
            distances.flatten()[i]
        ))

Recommendations for 2001: A Space Odyssey (1968):

1: Alien (1979), with distance of 0.3658532837052536:
2: Blade Runner (1982), with distance of 0.37560935880938506:
3: Aliens (1986), with distance of 0.3957780376034593:
4: Terminator, The (1984), with distance of 0.3965295810843581:
5: Star Wars: Episode V - The Empire Strikes Back (1980), with distance of 0.41133525377307567:
